# Step 1 — Safety Gate: OIG LEIE + PECOS Exclusion Check

**Business question:** Given a list of provider NPIs, should each provider be allowed into the Meroka marketplace?

**Logic:** Match each NPI against the OIG LEIE exclusion list and the PECOS enrollment file. If a provider appears on LEIE (with no reinstatement), they fail the gate. If they are not found in PECOS (not enrolled in Medicare), we flag that too. Output is a binary `gate_result` column: `PASS` or `FAIL`.

**Data sources:**
- **OIG LEIE** — `exclusions.oig.hhs.gov` (full exclusion list CSV, updated monthly)
- **PECOS** — `data.cms.gov` Medicare Fee-for-Service Public Provider Enrollment (Jan 2026 extract)
- **Sample NPI list** — 200 NPIs from PECOS + 10 known-excluded NPIs from LEIE for validation

**Caveats:**
- NPDB is out of scope for this sprint (access requires registration)
- Older LEIE records may have NPI = `0000000000` (pre-NPI era). Name-based matching not implemented here.
- State medical board checks are a separate workstream

## Step 1 — Load Data

**What this does:** We're pulling in three files that we'll need:

1. **LEIE (List of Excluded Individuals/Entities)** — the government's master list of providers who've been kicked out of federal healthcare programs. If someone is on this list, any entity that pays them for federally reimbursable services faces real legal exposure. This is the core of our safety gate.

2. **PECOS (Provider Enrollment, Chain and Ownership System)** — CMS's enrollment file for providers who bill Medicare. If a provider is actively enrolled here, it's a good signal. If they're not, it doesn't automatically mean they're bad, they just might not bill Medicare. But it's worth flagging.

3. **Sample NPI list** — 210 provider NPIs to test our logic on. 200 are real enrolled providers from PECOS (should pass the gate), and 10 are providers we know are on the LEIE exclusion list (should fail the gate). This lets us prove the matching works.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import os

# Load LEIE exclusion list
leie = pd.read_csv("LEIE_UPDATED.csv", encoding="latin-1", dtype=str)
print(f"LEIE: {leie.shape[0]:,} rows, {leie.shape[1]} columns")

# Load PECOS enrollment file
pecos = pd.read_csv("PECOS_Enrollment.csv", encoding="latin-1", dtype=str)
print(f"PECOS: {pecos.shape[0]:,} rows, {pecos.shape[1]} columns")

# Load sample NPI list
sample = pd.read_csv("sample_npi_list.csv", dtype=str)
print(f"Sample NPIs: {sample.shape[0]} rows")
print(f"  - PECOS_ENROLLED: {(sample['SOURCE'] == 'PECOS_ENROLLED').sum()}")
print(f"  - LEIE_EXCLUDED:  {(sample['SOURCE'] == 'LEIE_EXCLUDED').sum()}")

LEIE: 82,749 rows, 18 columns


PECOS: 2,957,262 rows, 11 columns
Sample NPIs: 210 rows
  - PECOS_ENROLLED: 200
  - LEIE_EXCLUDED:  10


## Step 2 — Exploratory Data Analysis

**What this does:** Before we start matching anything, we need to understand what we're working with. This is the "open the box and look inside" step.

For each dataset we check:
- **How big is it?** (number of rows and columns)
- **What columns are in it?** (the schema)
- **How much data is missing?** (null rates per column, so we know what's reliable)
- **What types of providers are in it?** (distribution of categories, specialties, states)

Key finding from LEIE: only ~10% of excluded providers have an NPI on file. The other ~90% are older records from before the NPI system existed (pre-2007). For those, you'd need to match on name + address, which we're not doing in this sprint. This is a known gap worth flagging.

The charts give a visual sense of where exclusions concentrate (by type, by state, by provider category) and what the PECOS enrollment landscape looks like.

In [2]:
# --- LEIE EDA ---
print("=" * 60)
print("LEIE EXCLUSION LIST")
print("=" * 60)

print(f"\nShape: {leie.shape}")
print(f"\nColumns: {list(leie.columns)}")

print(f"\nNull rates:")
print((leie.isnull().sum() / len(leie) * 100).round(1).to_string())

print(f"\nData types:")
print(leie.dtypes.to_string())

# NPI quality
npi_zeros = (leie["NPI"] == "0000000000").sum()
npi_real = (leie["NPI"] != "0000000000").sum()
print(f"\nNPI quality:")
print(f"  Real NPI:     {npi_real:,} ({npi_real/len(leie)*100:.1f}%)")
print(f"  Zero/missing: {npi_zeros:,} ({npi_zeros/len(leie)*100:.1f}%)")

# Reinstatement status
still_excluded = (leie["REINDATE"] == "00000000").sum()
reinstated = (leie["REINDATE"] != "00000000").sum()
print(f"\nExclusion status:")
print(f"  Still excluded: {still_excluded:,}")
print(f"  Reinstated:     {reinstated:,}")

LEIE EXCLUSION LIST

Shape: (82749, 18)

Columns: ['LASTNAME', 'FIRSTNAME', 'MIDNAME', 'BUSNAME', 'GENERAL', 'SPECIALTY', 'UPIN', 'NPI', 'DOB', 'ADDRESS', 'CITY', 'STATE', 'ZIP', 'EXCLTYPE', 'EXCLDATE', 'REINDATE', 'WAIVERDATE', 'WVRSTATE']

Null rates:
LASTNAME        4.1
FIRSTNAME       4.1
MIDNAME        29.3
BUSNAME        95.9
GENERAL         0.0
SPECIALTY       4.9
UPIN           92.8
NPI             0.0
DOB             5.1
ADDRESS         0.0
CITY            0.0
STATE           0.0
ZIP             0.0
EXCLTYPE        0.0
EXCLDATE        0.0
REINDATE        0.0
WAIVERDATE      0.0
WVRSTATE      100.0

Data types:
LASTNAME      object
FIRSTNAME     object
MIDNAME       object
BUSNAME       object
GENERAL       object
SPECIALTY     object
UPIN          object
NPI           object
DOB           object
ADDRESS       object
CITY          object
STATE         object
ZIP           object
EXCLTYPE      object
EXCLDATE      object
REINDATE      object
WAIVERDATE    object
WVRSTATE      ob

In [3]:
# LEIE visualizations

# Top exclusion types
fig1 = px.bar(
    leie["EXCLTYPE"].value_counts().head(10).reset_index(),
    x="EXCLTYPE", y="count",
    title="LEIE: Top 10 Exclusion Types",
    labels={"EXCLTYPE": "Exclusion Type", "count": "Count"}
)
fig1.show()

# Top provider categories
fig2 = px.bar(
    leie["GENERAL"].value_counts().head(10).reset_index(),
    x="GENERAL", y="count",
    title="LEIE: Top 10 Provider Categories",
    labels={"GENERAL": "Category", "count": "Count"}
)
fig2.show()

# Exclusions by state
state_counts = leie["STATE"].value_counts().head(15).reset_index()
fig3 = px.bar(
    state_counts, x="STATE", y="count",
    title="LEIE: Top 15 States by Exclusion Count",
    labels={"STATE": "State", "count": "Exclusions"}
)
fig3.show()

In [4]:
# --- PECOS EDA ---
print("=" * 60)
print("PECOS ENROLLMENT FILE")
print("=" * 60)

print(f"\nShape: {pecos.shape}")
print(f"\nColumns: {list(pecos.columns)}")

print(f"\nNull rates:")
print((pecos.isnull().sum() / len(pecos) * 100).round(1).to_string())

print(f"\nData types:")
print(pecos.dtypes.to_string())

# Unique NPIs vs total rows (providers can have multiple enrollments)
unique_npis = pecos["NPI"].nunique()
print(f"\nUnique NPIs: {unique_npis:,} (out of {len(pecos):,} enrollment rows)")
print(f"Avg enrollments per NPI: {len(pecos)/unique_npis:.1f}")

# Top provider types
print(f"\nTop 10 provider types:")
print(pecos["PROVIDER_TYPE_DESC"].value_counts().head(10).to_string())

PECOS ENROLLMENT FILE

Shape: (2957262, 11)

Columns: ['NPI', 'MULTIPLE_NPI_FLAG', 'PECOS_ASCT_CNTL_ID', 'ENRLMT_ID', 'PROVIDER_TYPE_CD', 'PROVIDER_TYPE_DESC', 'STATE_CD', 'FIRST_NAME', 'MDL_NAME', 'LAST_NAME', 'ORG_NAME']

Null rates:
NPI                    0.0
MULTIPLE_NPI_FLAG      0.0
PECOS_ASCT_CNTL_ID     0.0
ENRLMT_ID              0.0
PROVIDER_TYPE_CD       0.0
PROVIDER_TYPE_DESC     0.0
STATE_CD               0.0
FIRST_NAME            14.7
MDL_NAME              46.8
LAST_NAME             14.7
ORG_NAME              85.3

Data types:
NPI                   object
MULTIPLE_NPI_FLAG     object
PECOS_ASCT_CNTL_ID    object
ENRLMT_ID             object
PROVIDER_TYPE_CD      object
PROVIDER_TYPE_DESC    object
STATE_CD              object
FIRST_NAME            object
MDL_NAME              object
LAST_NAME             object
ORG_NAME              object



Unique NPIs: 2,542,195 (out of 2,957,262 enrollment rows)
Avg enrollments per NPI: 1.2

Top 10 provider types:
PROVIDER_TYPE_DESC
PRACTITIONER - NURSE PRACTITIONER                               404455
PART B SUPPLIER - CLINIC/GROUP PRACTICE                         238261
PRACTITIONER - PHYSICIAN ASSISTANT                              192960
PRACTITIONER - INTERNAL MEDICINE                                144014
PRACTITIONER - FAMILY PRACTICE                                  128953
PRACTITIONER - PHYSICAL THERAPIST IN PRIVATE PRACTICE           124790
PRACTITIONER - CLINICAL SOCIAL WORKER                           101444
PRACTITIONER - EMERGENCY MEDICINE                                84177
PRACTITIONER - CERTIFIED REGISTERED NURSE ANESTHETIST (CRNA)     79375
PRACTITIONER - DIAGNOSTIC RADIOLOGY                              64883


In [5]:
# PECOS visualizations

# Provider type distribution
fig4 = px.bar(
    pecos["PROVIDER_TYPE_DESC"].value_counts().head(15).reset_index(),
    x="PROVIDER_TYPE_DESC", y="count",
    title="PECOS: Top 15 Provider Types",
    labels={"PROVIDER_TYPE_DESC": "Provider Type", "count": "Enrollments"}
)
fig4.update_layout(xaxis_tickangle=-45)
fig4.show()

# Enrollments by state
fig5 = px.bar(
    pecos["STATE_CD"].value_counts().head(15).reset_index(),
    x="STATE_CD", y="count",
    title="PECOS: Top 15 States by Enrollment Count",
    labels={"STATE_CD": "State", "count": "Enrollments"}
)
fig5.show()

## Step 3 — Data Preprocessing & Cleaning

**What this does:** Get the data into shape before we start matching.

Three things happen here:

1. **Clean up the LEIE list.** The file uses `00000000` to mean "not reinstated" (still excluded) and `0000000000` to mean "no NPI on record." We filter down to only providers who are currently excluded AND have a real NPI. That takes us from ~82K records down to ~8K usable exclusion records.

2. **Build lookup sets.** We turn the cleaned LEIE NPIs into a fast-lookup set of excluded providers, and the PECOS NPIs into a fast-lookup set of enrolled providers. This is how the matching in the next step happens instantly, even against millions of records.

3. **Clean the sample list.** Strip any whitespace from NPIs so the matching is exact.

In [6]:
# --- Clean LEIE ---
# Strip whitespace from NPI column
leie["NPI"] = leie["NPI"].str.strip()

# Filter to currently excluded only (REINDATE == "00000000" means not reinstated)
leie_active = leie[leie["REINDATE"] == "00000000"].copy()
print(f"LEIE active exclusions: {len(leie_active):,} (dropped {len(leie) - len(leie_active):,} reinstated)")

# Filter to records with real NPIs (not 0000000000)
leie_with_npi = leie_active[leie_active["NPI"] != "0000000000"].copy()
print(f"LEIE with real NPI: {len(leie_with_npi):,} (dropped {len(leie_active) - len(leie_with_npi):,} without NPI)")

# Build the exclusion set: unique NPIs currently on the LEIE
excluded_npis = set(leie_with_npi["NPI"].unique())
print(f"Unique excluded NPIs: {len(excluded_npis):,}")

LEIE active exclusions: 82,749 (dropped 0 reinstated)
LEIE with real NPI: 8,482 (dropped 74,267 without NPI)
Unique excluded NPIs: 8,306


In [7]:
# --- Clean PECOS ---
pecos["NPI"] = pecos["NPI"].str.strip()

# Build enrolled NPI set (any NPI that appears in PECOS = actively enrolled in Medicare)
enrolled_npis = set(pecos["NPI"].unique())
print(f"Unique enrolled NPIs in PECOS: {len(enrolled_npis):,}")

# --- Clean sample ---
sample["NPI"] = sample["NPI"].str.strip()
print(f"\nSample NPI list: {len(sample)} providers")
print(sample.groupby("SOURCE").size())

Unique enrolled NPIs in PECOS: 2,542,195

Sample NPI list: 210 providers
SOURCE
LEIE_EXCLUDED      10
PECOS_ENROLLED    200
dtype: int64


## Step 4 — Core Transformation: Safety Gate Logic

**What this does:** This is the actual gate. For each provider NPI in our sample list, we ask two questions:

1. **Is this provider on the federal exclusion list (LEIE)?** If yes, they're flagged. Any employer directing patients to an excluded provider has legal exposure.

2. **Is this provider enrolled in Medicare (PECOS)?** This is informational, not a gate blocker. A provider not in PECOS might just be a cash-pay practice or only take commercial insurance. But it's useful context.

The output is a new column called `gate_result`: **PASS** or **FAIL**. Right now, FAIL means the provider is on LEIE. That's it. Simple binary.

For providers who fail, we also pull in the details from LEIE: what type of exclusion, when it happened, and their listed specialty. This matters for downstream reporting.

In [8]:
# Match each sample NPI against LEIE and PECOS
results = sample.copy()

# LEIE check: is this NPI on the active exclusion list?
results["leie_excluded"] = results["NPI"].isin(excluded_npis)

# PECOS check: is this NPI enrolled in Medicare?
results["pecos_enrolled"] = results["NPI"].isin(enrolled_npis)

# Gate result: FAIL if on LEIE, PASS otherwise
results["gate_result"] = np.where(results["leie_excluded"], "FAIL", "PASS")

# Add LEIE details for excluded providers (exclusion type, date)
leie_details = leie_with_npi[["NPI", "EXCLTYPE", "EXCLDATE", "GENERAL", "SPECIALTY"]].drop_duplicates(subset="NPI", keep="first")
results = results.merge(
    leie_details.rename(columns={
        "EXCLTYPE": "leie_excl_type",
        "EXCLDATE": "leie_excl_date",
        "GENERAL": "leie_provider_category",
        "SPECIALTY": "leie_specialty"
    }),
    on="NPI",
    how="left"
)

print(f"Gate results:")
print(results["gate_result"].value_counts())
print(f"\nPECOS enrollment among sample:")
print(results["pecos_enrolled"].value_counts())

Gate results:
gate_result
PASS    200
FAIL     10
Name: count, dtype: int64

PECOS enrollment among sample:
pecos_enrolled
True     200
False     10
Name: count, dtype: int64


In [9]:
# View the full results table
results[["NPI", "FIRST_NAME", "LAST_NAME", "STATE", "SOURCE", "leie_excluded", "pecos_enrolled", "gate_result", "leie_excl_type", "leie_excl_date"]].to_string()
results.head(20)

,NPI,FIRST_NAME,LAST_NAME,STATE,PROVIDER_TYPE,SOURCE,leie_excluded,pecos_enrolled,gate_result,leie_excl_type,leie_excl_date,leie_provider_category,leie_specialty
0,1164502100,NICOLE,WILLIAMS,TX,PRACTITIONER - REGISTERED DIETITIAN OR NUTRITI...,PECOS_ENROLLED,False,True,PASS,NaN,NaN,NaN,NaN
1,1285658807,KANAKAVALLI,IYER,CA,PRACTITIONER - ANESTHESIOLOGY,PECOS_ENROLLED,False,True,PASS,NaN,NaN,NaN,NaN
2,1083612303,JEROME,LOPEZ,TX,PRACTITIONER - NEUROLOGY,PECOS_ENROLLED,False,True,PASS,NaN,NaN,NaN,NaN
3,1063483899,KIMBERLY,FUSELIER,CA,PRACTITIONER - PHYSICAL THERAPIST IN PRIVATE P...,PECOS_ENROLLED,False,True,PASS,NaN,NaN,NaN,NaN
4,1841253606,YONG,HAHN,NY,PRACTITIONER - DIAGNOSTIC RADIOLOGY,PECOS_ENROLLED,False,True,PASS,NaN,NaN,NaN,NaN
5,1477548519,KEITH,WEBB,WV,PRACTITIONER - CERTIFIED REGISTERED NURSE ANES...,PECOS_ENROLLED,False,True,PASS,NaN,NaN,NaN,NaN
6,1164450557,MICHAEL,GOLDSTEIN,NY,PRACTITIONER - PSYCHIATRY,PECOS_ENROLLED,False,True,PASS,NaN,NaN,NaN,NaN
7,1710930052,GERSON,BERNHARD,CA,PRACTITIONER - RHEUMATOLOGY,PECOS_ENROLLED,False,True,PASS,NaN,NaN,NaN,NaN
8,1568548758,ELIZABETH,HAGEE,TX,PRACTITIONER - PEDIATRIC MEDICINE,PECOS_ENROLLED,False,True,PASS,NaN,NaN,NaN,NaN
9,1023239118,TOVA,STEINHAUSER,CA,PRACTITIONER - FAMILY PRACTICE,PECOS_ENROLLED,False,True,PASS,NaN,NaN,NaN,NaN


## Step 5 — Proof of Correctness

**What this does:** We need to prove the gate actually works before handing it off to engineering. We run three tests:

1. **Do the known-bad providers get caught?** We intentionally seeded 10 NPIs that we know are on the LEIE exclusion list. If all 10 come back as FAIL, the gate is working.

2. **Do the known-good providers pass?** The 200 NPIs we pulled from PECOS (actively enrolled Medicare providers) should overwhelmingly come back as PASS. If a bunch of them fail, something is wrong with our matching logic.

3. **Spot-check a specific case.** We take one excluded provider (Criselda Abad-Santos, NPI 1760461826) and trace through everything: does our output match the raw LEIE data? Same name, same exclusion type, same date? This is the "trust but verify" step.

If all three tests pass, the gate logic is validated end-to-end.

In [10]:
# --- Validation 1: All known-excluded NPIs should FAIL ---
excluded_results = results[results["SOURCE"] == "LEIE_EXCLUDED"]
all_excluded_fail = (excluded_results["gate_result"] == "FAIL").all()
print(f"TEST 1: All {len(excluded_results)} known-excluded NPIs flagged as FAIL?")
print(f"  Result: {'PASS ✓' if all_excluded_fail else 'FAIL ✗'}")
print(f"  Details:")
print(excluded_results[["NPI", "FIRST_NAME", "LAST_NAME", "gate_result", "leie_excl_type", "leie_excl_date"]].to_string(index=False))

print()

# --- Validation 2: PECOS-sourced NPIs should mostly PASS ---
pecos_results = results[results["SOURCE"] == "PECOS_ENROLLED"]
pecos_pass_rate = (pecos_results["gate_result"] == "PASS").mean()
pecos_fail_count = (pecos_results["gate_result"] == "FAIL").sum()
print(f"TEST 2: PECOS-sourced NPIs gate results")
print(f"  PASS rate: {pecos_pass_rate*100:.1f}%")
print(f"  FAIL count: {pecos_fail_count} (these are providers enrolled in Medicare but also on LEIE)")
if pecos_fail_count > 0:
    print(f"  Failed NPIs from PECOS sample:")
    print(pecos_results[pecos_results["gate_result"] == "FAIL"][["NPI", "FIRST_NAME", "LAST_NAME", "leie_excl_type"]].to_string(index=False))

print()

# --- Validation 3: Spot-check a specific excluded NPI ---
test_npi = "1760461826"  # CRISELDA ABAD-SANTOS
print(f"TEST 3: Spot-check NPI {test_npi}")
spot = results[results["NPI"] == test_npi].iloc[0]
print(f"  Name: {spot['FIRST_NAME']} {spot['LAST_NAME']}")
print(f"  LEIE excluded: {spot['leie_excluded']}")
print(f"  Exclusion type: {spot['leie_excl_type']}")
print(f"  Exclusion date: {spot['leie_excl_date']}")
print(f"  PECOS enrolled: {spot['pecos_enrolled']}")
print(f"  Gate result: {spot['gate_result']}")

# Cross-reference with raw LEIE data
raw_match = leie[leie["NPI"] == test_npi]
print(f"\n  Raw LEIE record for this NPI:")
print(raw_match[["LASTNAME", "FIRSTNAME", "NPI", "EXCLTYPE", "EXCLDATE", "REINDATE"]].to_string(index=False))

TEST 1: All 10 known-excluded NPIs flagged as FAIL?
  Result: PASS ✓
  Details:
       NPI FIRST_NAME   LAST_NAME gate_result leie_excl_type leie_excl_date
1760461826   CRISELDA ABAD-SANTOS        FAIL         1128b4       20250120
1477537496   JAMSHEED       ABADI        FAIL         1128b4       20140520
1124292966    CRISPIN  ABARIENTOS        FAIL         1128a1       20200618
1376108431      SHAFI       ABBAS        FAIL         1128a1       20251020
1194807255      JADAN     ABBASSI        FAIL         1128b4       20180620
1225029184     SORAYA   ABBASSIAN        FAIL         1128a2       20140420
1568528610     SAMUEL      ABBOTT        FAIL         1128b4       20240520
1912929787    TIMOTHY      ABBOTT        FAIL         1128a4       20220720
1669594099     HASSAN    ABDALLAH        FAIL         1128a1       20250820
1033550207      SAMIA  ABDELMAGID        FAIL         1128a1       20231220

TEST 2: PECOS-sourced NPIs gate results
  PASS rate: 100.0%
  FAIL count: 0 (these 

## Output Schema

The `results` dataframe is the deliverable. One row per NPI, with:

| Column | Type | Description |
|--------|------|-------------|
| `NPI` | str | 10-digit National Provider Identifier |
| `FIRST_NAME` | str | Provider first name |
| `LAST_NAME` | str | Provider last name |
| `STATE` | str | State code |
| `PROVIDER_TYPE` | str | Provider type description |
| `SOURCE` | str | Where this NPI came from in our sample |
| `leie_excluded` | bool | True if NPI is on the active LEIE exclusion list |
| `pecos_enrolled` | bool | True if NPI appears in PECOS Medicare enrollment |
| `gate_result` | str | **PASS** or **FAIL** — the safety gate decision |
| `leie_excl_type` | str | OIG exclusion type code (if excluded) |
| `leie_excl_date` | str | Date of exclusion, YYYYMMDD (if excluded) |
| `leie_provider_category` | str | LEIE provider category (if excluded) |
| `leie_specialty` | str | LEIE specialty (if excluded) |